# 🔬 Notebook 05 — Advanced Analytics (D6)
**Bluestock Fintech | Day 6**

Covers: VaR · Rolling Sharpe · Cohort Analysis · SIP Continuity · Recommender · Sector HHI · Monte Carlo · Efficient Frontier

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats, optimize
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE  = Path('..').resolve()
RAW   = BASE/'data'/'raw'
PROC  = BASE/'data'/'processed'
RF_D  = 0.065/252; TDAYS = 252

df_nav  = pd.read_csv(RAW/'02_nav_history.csv', parse_dates=['date'])
df_fund = pd.read_csv(RAW/'01_fund_master.csv').drop_duplicates('amfi_code')
df_tx   = pd.read_csv(RAW/'08_investor_transactions.csv', parse_dates=['transaction_date'])
df_port = pd.read_csv(RAW/'09_portfolio_holdings.csv')
df_perf = pd.read_csv(PROC/'fund_metrics.csv') if (PROC/'fund_metrics.csv').exists() else pd.DataFrame()
print("Data loaded ✓")

## 1. Historical VaR & CVaR (95%) per Fund

In [ ]:
var_rows = []
for code, grp in df_nav.groupby('amfi_code'):
    ret = grp.sort_values('date')['nav'].pct_change().dropna()
    if len(ret) < 60: continue
    var95  = np.percentile(ret, 5)
    cvar95 = ret[ret <= var95].mean()
    var99  = np.percentile(ret, 1)
    var_rows.append({'amfi_code':code,
                     'var_95_daily_pct' : round(var95*100,3),
                     'cvar_95_daily_pct': round(cvar95*100,3),
                     'var_99_daily_pct' : round(var99*100,3),
                     'var_95_ann_pct'   : round(var95*np.sqrt(TDAYS)*100,2)})

df_var = pd.DataFrame(var_rows)
df_var.to_csv(PROC/'var_cvar_report.csv', index=False)

# Plot VaR distribution for one fund
code_example = df_var['amfi_code'].iloc[0]
ret_ex = df_nav[df_nav['amfi_code']==code_example].sort_values('date')['nav'].pct_change().dropna()
fig, ax = plt.subplots(figsize=(10,4))
ax.hist(ret_ex*100, bins=80, color='steelblue', alpha=0.7, edgecolor='white')
var_ex = df_var[df_var['amfi_code']==code_example]['var_95_daily_pct'].values[0]
ax.axvline(var_ex, color='red', linestyle='--', linewidth=2, label=f'VaR 95%: {var_ex:.2f}%')
ax.axvline(df_var[df_var['amfi_code']==code_example]['cvar_95_daily_pct'].values[0],
           color='darkred', linestyle=':', linewidth=2,
           label=f"CVaR 95%: {df_var[df_var['amfi_code']==code_example]['cvar_95_daily_pct'].values[0]:.2f}%")
ax.set_title(f'Daily Return Distribution & VaR — Fund {code_example}', fontsize=12, fontweight='bold')
ax.set_xlabel('Daily Return (%)'); ax.set_ylabel('Frequency')
ax.legend(); plt.tight_layout()
plt.savefig(PROC/'chart_var_distribution.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"\nVaR Report (worst 5 funds):")
print(df_var.nsmallest(5,'var_95_daily_pct').to_string(index=False))

## 2. Rolling 90-Day Sharpe Ratio

In [ ]:
sample_codes = df_nav['amfi_code'].unique()[:5]
fig, ax = plt.subplots(figsize=(13,5))

for i, code in enumerate(sample_codes):
    sub = df_nav[df_nav['amfi_code']==code].sort_values('date')
    ret = sub['nav'].pct_change().dropna()
    rolling_sharpe = (
        ret.rolling(90).mean() - RF_D
    ) / ret.rolling(90).std() * np.sqrt(TDAYS)
    ax.plot(sub['date'].iloc[1:], rolling_sharpe.values,
            label=str(code), alpha=0.85, linewidth=1.4)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axhline(1, color='green', linewidth=0.8, linestyle=':', label='Sharpe=1')
ax.set_title('Rolling 90-Day Sharpe Ratio (5 Funds)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Rolling Sharpe Ratio')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(PROC/'rolling_sharpe_chart.png', dpi=130, bbox_inches='tight')
plt.show()
print("Rolling Sharpe chart saved ✓")

## 3. Investor Cohort Analysis

In [ ]:
df_tx['year'] = df_tx['transaction_date'].dt.year
first_tx = df_tx.groupby('investor_id')['transaction_date'].min().reset_index()
first_tx['cohort_year'] = first_tx['transaction_date'].dt.year

df_tx_c = df_tx.merge(first_tx[['investor_id','cohort_year']], on='investor_id')
cohort = df_tx_c[df_tx_c['transaction_type']=='Sip'].groupby('cohort_year').agg(
    num_investors   = ('investor_id','nunique'),
    avg_sip_amount  = ('amount_inr','mean'),
    total_invested  = ('amount_inr','sum'),
    num_transactions= ('investor_id','count'),
).reset_index()
cohort['avg_tx_per_investor'] = (cohort['num_transactions'] / cohort['num_investors']).round(1)
cohort['total_invested_crore'] = (cohort['total_invested']/1e7).round(2)
cohort.to_csv(PROC/'cohort_analysis.csv', index=False)

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].bar(cohort['cohort_year'].astype(str), cohort['avg_sip_amount'],
            color=['#1565C0','#2E7D32','#C62828','#F57F17'])
axes[0].set_title('Avg SIP Amount by Cohort Year', fontsize=11, fontweight='bold')
axes[0].set_xlabel('First Investment Year'); axes[0].set_ylabel('Avg SIP (Rs.)')

axes[1].bar(cohort['cohort_year'].astype(str), cohort['avg_tx_per_investor'],
            color=['#1565C0','#2E7D32','#C62828','#F57F17'])
axes[1].set_title('Avg Transactions per Investor by Cohort', fontsize=11, fontweight='bold')
axes[1].set_xlabel('First Investment Year')

plt.tight_layout()
plt.savefig(PROC/'chart_cohort.png', dpi=130, bbox_inches='tight')
plt.show()
print(cohort.to_string(index=False))

## 4. SIP Continuity — At-Risk Investors

In [ ]:
sip_only = df_tx[df_tx['transaction_type']=='Sip'].copy()
sip_only = sip_only.sort_values(['investor_id','transaction_date'])
sip_only['gap_days'] = sip_only.groupby('investor_id')['transaction_date'].diff().dt.days

continuity = sip_only.groupby('investor_id').agg(
    num_sips    = ('transaction_date','count'),
    avg_gap     = ('gap_days','mean'),
    max_gap     = ('gap_days','max'),
    total_invested = ('amount_inr','sum'),
).reset_index()
continuity = continuity[continuity['num_sips'] >= 6]
continuity['at_risk'] = continuity['max_gap'] > 35

at_risk_count = continuity['at_risk'].sum()
print(f"Total SIP investors (6+ transactions): {len(continuity):,}")
print(f"At-risk investors (gap > 35 days):     {at_risk_count:,} ({at_risk_count/len(continuity)*100:.1f}%)")
continuity.to_csv(PROC/'sip_continuity.csv', index=False)

fig, ax = plt.subplots(figsize=(8,4))
continuity['avg_gap'].hist(bins=40, color='steelblue', alpha=0.7, ax=ax, edgecolor='white')
ax.axvline(35, color='red', linestyle='--', label='35-day threshold')
ax.set_title('Distribution of Average SIP Gap (Days)', fontsize=12, fontweight='bold')
ax.set_xlabel('Avg Gap (Days)'); ax.set_ylabel('No. of Investors')
ax.legend(); plt.tight_layout()
plt.savefig(PROC/'chart_sip_continuity.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. Fund Recommender

In [ ]:
import sys
sys.path.insert(0, str(BASE/'scripts'))
from recommender import recommend, herfindahl_hirschman_index

for risk in ['Low','Moderate','High']:
    print(f"\n{'='*55}")
    print(f"  Risk: {risk} | Horizon: Long (5yr)")
    print('='*55)
    recs = recommend(risk_appetite=risk, horizon='long', top_n=3)
    if not recs.empty:
        print(recs[['scheme_name','sub_category','sharpe_ratio','expense_ratio_pct']].to_string())
    else:
        print("  No matching funds.")

## 6. Sector Concentration (HHI)

In [ ]:
hhi_df = herfindahl_hirschman_index(df_port)
hhi_df.to_csv(PROC/'sector_hhi.csv', index=False)

fig, ax = plt.subplots(figsize=(10,4))
colors_hhi = ['#2E7D32' if c=='Diversified' else '#F57F17' if c=='Moderate' else '#C62828'
               for c in hhi_df['concentration']]
ax.bar(hhi_df['amfi_code'].astype(str), hhi_df['sector_hhi'], color=colors_hhi)
ax.axhline(0.10, color='orange', linestyle='--', label='Moderate (HHI=0.10)')
ax.axhline(0.18, color='red',    linestyle='--', label='Concentrated (HHI=0.18)')
ax.set_title('Sector HHI by Fund (Portfolio Concentration Risk)', fontsize=12, fontweight='bold')
ax.set_xlabel('AMFI Code'); ax.set_ylabel('HHI Score')
ax.legend(); plt.tight_layout()
plt.savefig(PROC/'chart_hhi.png', dpi=130, bbox_inches='tight')
plt.show()
print(hhi_df.to_string(index=False))

## 7. Monte Carlo NAV Projection (B3)

In [ ]:
# 5-year Monte Carlo for top 3 funds
code_mc = df_nav['amfi_code'].unique()[:3]
n_sim = 500; n_days = 252*5

fig, axes = plt.subplots(1, len(code_mc), figsize=(14, 5))
for idx, code in enumerate(code_mc):
    sub = df_nav[df_nav['amfi_code']==code].sort_values('date')
    nav = sub['nav'].values
    ret = np.diff(np.log(nav))
    mu  = ret.mean(); sigma = ret.std()
    last_nav = nav[-1]
    sims = np.zeros((n_days, n_sim))
    for s in range(n_sim):
        shocks = np.random.normal(mu, sigma, n_days)
        sims[:, s] = last_nav * np.exp(np.cumsum(shocks))
    p5  = np.percentile(sims, 5,  axis=1)
    p50 = np.percentile(sims, 50, axis=1)
    p95 = np.percentile(sims, 95, axis=1)
    t   = np.arange(n_days)
    ax  = axes[idx]
    ax.fill_between(t, p5, p95, alpha=0.2, color='steelblue', label='90% CI')
    ax.plot(t, p50, color='steelblue', linewidth=2, label='Median')
    ax.plot(t, p5,  color='red',   linewidth=1, linestyle='--', label='5th pct')
    ax.plot(t, p95, color='green', linewidth=1, linestyle='--', label='95th pct')
    ax.set_title(f'Monte Carlo — Fund {code}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Trading Days'); ax.set_ylabel('Projected NAV (Rs.)')
    ax.legend(fontsize=7)

plt.suptitle('5-Year Monte Carlo NAV Projection (500 Simulations)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROC/'chart_monte_carlo.png', dpi=130, bbox_inches='tight')
plt.show()
print("Monte Carlo chart saved ✓")

## 8. Markowitz Efficient Frontier (B4)

In [ ]:
# Select 5 equity funds for portfolio optimisation
ef_codes = df_nav['amfi_code'].unique()[:5]
pivot = df_nav[df_nav['amfi_code'].isin(ef_codes)].pivot_table(
    index='date', columns='amfi_code', values='nav')
returns = pivot.pct_change().dropna()
mu_vec  = returns.mean() * TDAYS
cov_mat = returns.cov()  * TDAYS
n_assets = len(ef_codes)

def portfolio_stats(weights):
    port_ret = np.dot(weights, mu_vec)
    port_vol = np.sqrt(weights @ cov_mat.values @ weights)
    sharpe   = (port_ret - 0.065) / port_vol
    return port_ret, port_vol, sharpe

# Monte Carlo portfolios
n_portfolios = 3000
p_rets, p_vols, p_sharpes = [], [], []
for _ in range(n_portfolios):
    w = np.random.dirichlet(np.ones(n_assets))
    r, v, s = portfolio_stats(w)
    p_rets.append(r); p_vols.append(v); p_sharpes.append(s)

# Min-volatility & Max-Sharpe
constraints = ({'type':'eq','fun': lambda w: w.sum()-1})
bounds = [(0.05,0.60)]*n_assets
min_vol = optimize.minimize(lambda w: portfolio_stats(w)[1],
    np.ones(n_assets)/n_assets, method='SLSQP', bounds=bounds, constraints=constraints)
max_sharpe = optimize.minimize(lambda w: -portfolio_stats(w)[2],
    np.ones(n_assets)/n_assets, method='SLSQP', bounds=bounds, constraints=constraints)

fig, ax = plt.subplots(figsize=(10,7))
sc = ax.scatter(p_vols, p_rets, c=p_sharpes, cmap='viridis', alpha=0.5, s=8)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')
mv_r, mv_v, _ = portfolio_stats(min_vol.x)
ms_r, ms_v, _ = portfolio_stats(max_sharpe.x)
ax.scatter(mv_v, mv_r, color='blue',  s=200, marker='*', zorder=5, label=f'Min Vol  (ret={mv_r*100:.1f}%)')
ax.scatter(ms_v, ms_r, color='red',   s=200, marker='*', zorder=5, label=f'Max Sharpe (ret={ms_r*100:.1f}%)')
ax.set_title('Markowitz Efficient Frontier (5 Equity Funds)', fontsize=13, fontweight='bold')
ax.set_xlabel('Annualised Volatility'); ax.set_ylabel('Annualised Return')
ax.legend()
plt.tight_layout()
plt.savefig(PROC/'chart_efficient_frontier.png', dpi=130, bbox_inches='tight')
plt.show()

print("\nMax Sharpe Portfolio Weights:")
for code, w in zip(ef_codes, max_sharpe.x):
    print(f"  Fund {code}: {w*100:.1f}%")
print("\n✅ Advanced Analytics Complete!")